# Customer Segmentation Analysis

This notebook generates a synthetic customer dataset, performs clustering using K-Means, and analyzes segment characteristics based on behavior and demographics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import silhouette_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

sns.set(style='whitegrid', palette='muted')
pd.options.display.max_columns = None

In [ ]:
# Generate a synthetic customer dataset
rng = np.random.default_rng(2026)
n_customers = 200
ages = rng.integers(18, 70, size=n_customers)
genders = rng.choice(['Male', 'Female'], size=n_customers, p=[0.48, 0.52])
income = rng.normal(65000, 25000, size=n_customers).clip(18000, 160000).round(0)
purchases_per_month = rng.poisson(3.5, size=n_customers).clip(1, 12)
avg_order_value = (rng.normal(80, 35, size=n_customers) + 0.5 * purchases_per_month).clip(10, 320).round(2)
spend_score = rng.integers(20, 100, size=n_customers)
tenure_months = rng.integers(2, 120, size=n_customers)
loyalty_score = np.clip((0.45 * tenure_months + 0.35 * spend_score + rng.normal(0, 10, n_customers)), 10, 100).round(1)
recency_days = rng.integers(1, 90, size=n_customers)
product_category = rng.choice(['Electronics', 'Apparel', 'Home & Garden', 'Beauty', 'Sports'], size=n_customers, p=[0.22, 0.28, 0.2, 0.16, 0.14])

customers = pd.DataFrame({
    'CustomerID': np.arange(1, n_customers + 1),
    'Age': ages,
    'Gender': genders,
    'AnnualIncome': income,
    'PurchasesPerMonth': purchases_per_month,
    'AvgOrderValue': avg_order_value,
    'SpendScore': spend_score,
    'TenureMonths': tenure_months,
    'LoyaltyScore': loyalty_score,
    'RecencyDays': recency_days,
    'ProductCategory': product_category
})
customers.head()

In [ ]:
customers.describe(include='all').T

## Preprocessing and Feature Selection

We create a pipeline to scale numerical features and one-hot encode categorical fields for clustering.

In [ ]:
features = ['Age', 'Gender', 'AnnualIncome', 'PurchasesPerMonth', 'AvgOrderValue', 'SpendScore', 'TenureMonths', 'LoyaltyScore', 'RecencyDays', 'ProductCategory']
numeric_features = ['Age', 'AnnualIncome', 'PurchasesPerMonth', 'AvgOrderValue', 'SpendScore', 'TenureMonths', 'LoyaltyScore', 'RecencyDays']
categorical_features = ['Gender', 'ProductCategory']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first'), categorical_features)
], remainder='drop')

pipeline = Pipeline([
    ('preprocessor', preprocessor),
])

X = pipeline.fit_transform(customers[features])
X.shape

## Finding the Best Number of Segments

We evaluate different values of `k` using the elbow method and silhouette score.

In [ ]:
inertia = []
silhouette = []
k_candidates = range(2, 8)

for k in k_candidates:
    model = KMeans(n_clusters=k, random_state=2026, n_init=10)
    labels = model.fit_predict(X)
    inertia.append(model.inertia_)
    silhouette.append(silhouette_score(X, labels))

results = pd.DataFrame({'k': list(k_candidates), 'Inertia': inertia, 'Silhouette': silhouette})
results

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()
ax1.plot(results['k'], results['Inertia'], marker='o', label='Inertia', color='#247ba0')
ax2.plot(results['k'], results['Silhouette'], marker='o', label='Silhouette', color='#ff7f51')
ax1.set_xlabel('Number of clusters (k)')
ax1.set_ylabel('Inertia', color='#247ba0')
ax2.set_ylabel('Silhouette Score', color='#ff7f51')
ax1.set_xticks(results['k'])
fig.suptitle('Elbow Method and Silhouette Scores for KMeans')
fig.tight_layout()
plt.show()

## Apply K-Means and Profile Segments

Based on the analysis, we choose a good value of `k`. Typically `k=4` or `k=5` works for this synthetic dataset.

In [ ]:
k_optimal = 4
kmeans = KMeans(n_clusters=k_optimal, random_state=2026, n_init=20)
customers['Segment'] = kmeans.fit_predict(X)
customers['Segment'] = customers['Segment'].astype('category')
customers['Segment'].value_counts().sort_index()

In [ ]:
segment_profile = customers.groupby('Segment').agg({
    'Age': ['mean', 'min', 'max'],
    'AnnualIncome': ['mean', 'min', 'max'],
    'PurchasesPerMonth': 'mean',
    'AvgOrderValue': 'mean',
    'SpendScore': 'mean',
    'TenureMonths': 'mean',
    'LoyaltyScore': 'mean',
    'RecencyDays': 'mean',
    'CustomerID': 'count'
}).round(2)
segment_profile.columns = ['_'.join(col).strip() for col in segment_profile.columns.values]
segment_profile = segment_profile.rename(columns={'CustomerID_count': 'Count'})
segment_profile

## Visualizing Customer Segments

We visualize purchase behavior, income distribution, and segment sizes.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=customers, x='Segment', palette='Set2')
plt.title('Customer Count by Segment')
plt.xlabel('Segment')
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(data=customers, x='Segment', y='AnnualIncome', palette='Set2')
plt.title('Annual Income by Segment')
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(data=customers, x='Segment', y='SpendScore', palette='Set2')
plt.title('Spend Score by Segment')
plt.show()

In [ ]:
pair_features = ['AnnualIncome', 'PurchasesPerMonth', 'AvgOrderValue', 'SpendScore']
sns.pairplot(customers, vars=pair_features, hue='Segment', palette='Set2', diag_kind='hist', corner=True)
plt.suptitle('Pairwise Relationships by Segment', y=1.02)
plt.show()

## Summary and Insights

This segmentation approach reveals customer groups that differ by income, purchase frequency, average order value, and loyalty. Use these segments for targeted campaigns, personalized offers, and product recommendations.